# Vehicle Counter Integrate with Database

## Install semua package yang dibutuhkan (ada di requirements.txt)

In [ ]:
!pip install -r requirements.txt

## import semua packages

In [ ]:
import cv2
import os
import psycopg2
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
model = YOLO("yolov8n.pt") 



## Load Video Source 

In [ ]:
# load video source
video_path = 'https://dki-jkt.balitower.co.id:7028/502506_JKP_SATPOL-PP_JL.-GERBANG-PEMUDA-C04_CCTV-01/tracks-v1/index.fmp4.m3u8' 
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_display = cv2.resize(frame, (960, 540))
    
    cv2.imshow("CCTV Jakarta", frame_display)
    if cv2.waitKey(20) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()


## Connect ke Database

In [ ]:
load_dotenv(override=True)

try:
    conn = psycopg2.connect(
        host=os.getenv("DB_HOST"),
        database=os.getenv("DB_NAME"),  
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT")
    )
    cursor = conn.cursor()
    print(" Berhasil terhubung ke database PostgreSQL!")
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS traffic_detection_logs (
            id SERIAL PRIMARY KEY,
            vehicle_type VARCHAR(50) NOT NULL,
            confidence_score REAL NOT NULL,
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
    """)
    conn.commit()
    print(" Tabel 'traffic_detection_logs' sudah siap!")
except Exception as e:
    print(f"Gagal koneksi ke PostgreSQL: {e}")
    exit()


# Object Detection (Mobil, Motor, Bus, Truk)

In [ ]:
video_path = 'https://dki-jkt.balitower.co.id:7028/502506_JKP_SATPOL-PP_JL.-GERBANG-PEMUDA-C04_CCTV-01/tracks-v1/index.fmp4.m3u8'
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
class_map = {
    2: "Mobil",
    3: "Motor",
    5: "Bus",
    7: "Truk"
}
frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1
    results = model(frame, classes=[3, 2, 5, 7], verbose=False) # 5 = bus, 2 = car, 3 = motorcycle, 7 = truck
    annotated_frame = results[0].plot()

    if frame_count % 30 == 0:
         for box in results[0].boxes:
            cls_id = int(box.cls[0])
            conf = round(float(box.conf[0]) * 100, 1)  
            vehicle_name = class_map.get(cls_id, "Lainnya")
            check_query = """
                SELECT id FROM traffic_detection_logs 
                WHERE vehicle_type = %s AND ROUND(confidence_score::numeric, 1) = %s
                ORDER BY id DESC LIMIT 1;
            """
            cursor.execute(check_query, (vehicle_name, conf))
            existing_row = cursor.fetchone()
            if existing_row:
                log_id = existing_row[0]
                update_query = """
                    UPDATE traffic_detection_logs 
                    SET timestamp = CURRENT_TIMESTAMP 
                    WHERE id = %s;
                """
                cursor.execute(update_query, (log_id,))
                print(f"[DB UPDATE] ID #{log_id} ({vehicle_name} {conf}%) waktunya diperbarui.")
            else:
                insert_query = """
                    INSERT INTO traffic_detection_logs (vehicle_type, confidence_score)
                    VALUES (%s, %s);
                """
                cursor.execute(insert_query, (vehicle_name, conf))
                print(f"[DB INSERT] {vehicle_name} ({conf}%) baru tercatat!")
    conn.commit()
    cv2.imshow("YOLOv8 + PostgreSQL Real-Time Monitoring", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
if cursor:
    cursor.close()
    conn.close()
cap.release()
cv2.destroyAllWindows()
print("Koneksi database ditutup dan program selesai.")